# Rețele neuronale și deep learning — Seminarul 8: Modele generative: autoencodere, VAE, GAN, difuzie
Prof. dr. Daniel Traian Pele — Academia de Studii Economice din București (ASE); Academia Română, Institutul de Prognoză Economică (IPE)

[Deschide în Colab](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/main/notebooks/L8_seminar_ro.ipynb)

Sarcinile seminarului, pe datele reale din `../data/clean` (vezi `../data/SOURCES.md`). Scrieți codul în celulele goale de sub fiecare sarcină; păstrați valoarea fixă a generatorului aleator, ca rezultatele să poată fi comparate.

In [ ]:
# Google Colab: copiem repo-ul cursului, ca datele și rezultatele salvate să fie găsite
import os, subprocess, sys
if 'google.colab' in sys.modules and not os.getcwd().startswith('/content/NEURAL_BIZ'):
    if not os.path.isdir('/content/NEURAL_BIZ'):
        subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/danpele/NEURAL_BIZ.git', '/content/NEURAL_BIZ'], check=True)
    os.chdir('/content/NEURAL_BIZ/notebooks')

In [ ]:
import os, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
DATA = '../data/clean/'
SEED = 20261108
random.seed(SEED); np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
except ImportError:
    DEVICE = 'cpu'
plt.rcParams.update({'figure.facecolor': 'none', 'axes.facecolor': 'none', 'axes.grid': False,
                     'axes.spines.top': False, 'axes.spines.right': False, 'legend.frameon': False})
print('device:', DEVICE)

## Pregătire: datele de fraudă, împărțite în timp

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, precision_recall_curve
fr = pd.read_csv(DATA + 'creditcard_fraud_sample.csv').sort_values('Time').reset_index(drop=True)
cols = [f'V{i}' for i in range(1, 29)]
X = np.column_stack([fr[cols].values, np.log1p(fr['Amount'].values)]).astype(np.float32)
y, w = fr['Class'].values, fr['weight'].values
q60, q80 = np.quantile(fr['Time'], [0.6, 0.8])
itr, iva, ite = (fr.Time <= q60).values, ((fr.Time > q60) & (fr.Time <= q80)).values, (fr.Time > q80).values
leg = itr & (y == 0)
mu, sd = X[leg].mean(0), X[leg].std(0) + 1e-6
Xs = (X - mu) / sd
print('training legitimate:', leg.sum(), ' test rows:', ite.sum(), ' test frauds:', y[ite].sum())

In [ ]:
class AE(nn.Module):
    def __init__(self, d=29, k=8, h=16):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(d, h), nn.ReLU(), nn.Linear(h, k))
        self.dec = nn.Sequential(nn.ReLU(), nn.Linear(k, h), nn.ReLU(), nn.Linear(h, d))
    def forward(self, x):
        return self.dec(self.enc(x))

def train_ae(k=8, seed=1, epochs=60, patience=6):
    torch.manual_seed(seed)
    m = AE(k=k); opt = torch.optim.Adam(m.parameters(), lr=1e-3, weight_decay=1e-5)
    xt, xv = torch.tensor(Xs[leg]), torch.tensor(Xs[iva & (y == 0)])
    best, state, wait = 1e9, None, 0
    g = torch.Generator().manual_seed(seed)
    for ep in range(epochs):
        m.train(); perm = torch.randperm(len(xt), generator=g)
        for b in range(0, len(perm), 256):
            j = perm[b:b + 256]; loss = F.mse_loss(m(xt[j]), xt[j])
            opt.zero_grad(); loss.backward(); opt.step()
        m.eval()
        with torch.no_grad():
            vl = float(F.mse_loss(m(xv), xv))
        if vl < best - 1e-5:
            best, wait, state = vl, 0, {k_: v.clone() for k_, v in m.state_dict().items()}
        else:
            wait += 1
            if wait >= patience: break
    m.load_state_dict(state); m.eval()
    with torch.no_grad():
        return ((m(torch.tensor(Xs)) - torch.tensor(Xs)) ** 2).mean(1).numpy()

def wap(s):
    return average_precision_score(y[ite], s[ite], sample_weight=w[ite])



**Sarcina 1.** Antrenați autoencoderul cu $k\in\{4, 8, 16\}$ (sămînța 1) și raportați PR-AUC ponderat pe partea de test. Ce $k$ ați alege și pe ce date trebuie făcută alegerea?

In [ ]:
# codul vostru aici

**Sarcina 2.** O echipă antifraudă poate verifica primele 0,1 % scoruri din partea de test (număr ponderat). Cîte fraude găsește autoencoderul acolo?

In [ ]:
# codul vostru aici

## Pregătire: randamentele S&P 500

In [ ]:
from scipy import stats
x = pd.read_csv(DATA + 'markets_daily.csv', parse_dates=['date'])
px = x[x.ticker == 'SP500'].set_index('date')['adjclose'].dropna()
end = (px.index.max() + pd.Timedelta(days=1)).to_period('M').to_timestamp() - pd.Timedelta(days=1)
r = np.log(px.loc[:end]).diff().dropna()
rtr, rte = r.loc[:'2019-12-31'], r.loc['2020-01-01':]
s_tr = float(rtr.std()); ztr = (rtr.values / s_tr).astype(np.float32)


**Sarcina 3.** Estimați o distribuție Student-t pe randamentele de antrenare (2005–2019) cu `scipy.stats.t.fit` și calculați cuantila de 1 %. Comparați rata depășirilor din 2020 cu cea a normalei.

In [ ]:
# codul vostru aici

**Sarcina 4.** Gruparea volatilității: calculați autocorelația lui $|r_t|$ la decalajele 1, 5 și 20 pentru randamentele de antrenare și pentru aceleași randamente amestecate aleator. Ce îi lipsește unui generator de zile independente?

In [ ]:
# codul vostru aici

## Pregătire: curbele randamentelor din zona euro

In [ ]:
yc = pd.read_csv(DATA + 'ecb_yield_curve_daily.csv', parse_dates=['date']).set_index('date')
end = (yc.index.max() + pd.Timedelta(days=1)).to_period('M').to_timestamp() - pd.Timedelta(days=1)
MAT = ['y_3M', 'y_6M', 'y_1Y', 'y_2Y', 'y_3Y', 'y_5Y', 'y_7Y', 'y_10Y', 'y_15Y', 'y_20Y', 'y_30Y']
Y = yc.loc[:end, MAT].dropna().values.astype(np.float32); ym, ys = Y.mean(0), Y.std(0); Yz = (Y - ym) / ys
from scipy import stats
print(Y.shape)

**Sarcina 5.** Componentele principale ale curbelor standardizate: ce pondere din varianță explică primele două și trei componente? Interpretați primul vector de ponderi.

In [ ]:
# codul vostru aici

**Sarcina 6.** Generați 5.000 de curbe dintr-o normală multivariată estimată pe curbele standardizate. Ce pondere este inversată (10 ani sub 3 luni) și ce pondere are randamentul la 10 ani sub −0,5 %? Comparați cu datele reale.

In [ ]:
# codul vostru aici